In [2]:
import datetime as dt
import requests
import pandas as pd

# --- настройки ---
SECID = "SBER"  # поменяйте на нужный тикер, например: GAZP, LKOH, YNDX, TCSG

# Последние ~30 дней (календарные). Если нужны строго торговые дни — это тоже можно сделать.
till = dt.date.today()
from_ = till - dt.timedelta(days=30)

url = f"https://iss.moex.com/iss/engines/stock/markets/shares/securities/{SECID}/candles.json"
params = {
    "from": from_.isoformat(),
    "till": till.isoformat(),
    "interval": 24,   # 24 = дневные свечи
    "iss.meta": "off",
    "iss.only": "candles",
}

rows = []
start = 0
while True:
    r = requests.get(url, params={**params, "start": start}, timeout=30)
    r.raise_for_status()
    payload = r.json()["candles"]
    cols = payload["columns"]
    data = payload["data"]

    if not data:
        break

    rows.extend(data)
    start += len(data)

candles = pd.DataFrame(rows, columns=cols)
if candles.empty:
    raise RuntimeError(f"Нет данных по {SECID} за период {from_}..{till}. Проверьте SECID/рынок.")

# Средняя цена за месяц: среднее арифметическое дневных цен закрытия
candles["begin"] = pd.to_datetime(candles["begin"])
avg_close = float(pd.to_numeric(candles["close"], errors="coerce").dropna().mean())

print(f"{SECID}: средняя цена закрытия за последние 30 дней ({from_}..{till}) = {avg_close:.2f} RUB")
print(f"Дней в выборке: {len(candles)}")

candles[["begin", "open", "high", "low", "close", "volume"]].sort_values("begin").tail(10)

SBER: средняя цена закрытия за последние 30 дней (2026-03-07..2026-04-06) = 316.55 RUB
Дней в выборке: 27


,begin,open,high,low,close,volume
17,2026-03-28,313.90,314.26,313.31,314.13,1050754
18,2026-03-29,314.13,314.89,314.05,314.89,856008
19,2026-03-30,315.05,316.30,312.40,314.48,25240801
20,2026-03-31,315.00,315.63,313.43,314.22,13040353
21,2026-04-01,314.65,317.47,314.15,316.71,18962642
22,2026-04-02,316.75,317.21,315.18,316.27,9372317
23,2026-04-03,316.47,317.41,314.21,314.65,14790929
24,2026-04-04,315.03,315.44,314.64,315.12,1017258
25,2026-04-05,315.10,315.40,314.55,315.03,915253
26,2026-04-06,315.04,319.70,314.60,318.97,31063560


In [3]:
import numpy as np

# --- Датасет и модель: baseline прогноз "закрытие завтра выше, чем сегодня" ---
# Идея: берём дневные свечи нескольких акций за несколько лет, строим простые тех.признаки
# и обучаем классификатор предсказывать направление следующего дня.

TICKERS = [
    "SBER", "GAZP", "LKOH", "GMKN", "ROSN",
    "NVTK", "TATN", "MGNT", "ALRS", "VTBR",
]

def fetch_candles(secid: str, from_date: dt.date, till_date: dt.date, interval: int = 24) -> pd.DataFrame:
    url = f"https://iss.moex.com/iss/engines/stock/markets/shares/securities/{secid}/candles.json"
    params = {
        "from": from_date.isoformat(),
        "till": till_date.isoformat(),
        "interval": interval,
        "iss.meta": "off",
        "iss.only": "candles",
    }

    rows = []
    start = 0
    while True:
        r = requests.get(url, params={**params, "start": start}, timeout=30)
        r.raise_for_status()
        payload = r.json()["candles"]
        cols = payload["columns"]
        data = payload["data"]
        if not data:
            break
        rows.extend(data)
        start += len(data)

    df = pd.DataFrame(rows, columns=cols)
    if df.empty:
        return df

    df = df.copy()
    df["secid"] = secid
    df["begin"] = pd.to_datetime(df["begin"])
    for c in ["open", "high", "low", "close", "value", "volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def make_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["secid", "begin"]).copy()

    # базовые доходности
    df["ret1"] = df.groupby("secid")["close"].pct_change(1)
    df["ret2"] = df.groupby("secid")["close"].pct_change(2)
    df["ret5"] = df.groupby("secid")["close"].pct_change(5)

    # внутридневные признаки
    df["hl_spread"] = (df["high"] - df["low"]) / df["close"]
    df["co"] = (df["close"] - df["open"]) / df["open"]

    # объём
    df["vol_chg1"] = df.groupby("secid")["volume"].pct_change(1)

    # скользящие средние и волатильность
    g = df.groupby("secid")
    df["ma5"] = g["close"].transform(lambda s: s.rolling(5).mean())
    df["ma20"] = g["close"].transform(lambda s: s.rolling(20).mean())
    df["ma_ratio"] = df["ma5"] / df["ma20"] - 1.0

    df["vol5"] = g["ret1"].transform(lambda s: s.rolling(5).std())
    df["vol20"] = g["ret1"].transform(lambda s: s.rolling(20).std())
    df["vol_ratio"] = df["vol5"] / df["vol20"]

    # таргет: завтра close выше, чем сегодня
    df["close_next"] = g["close"].shift(-1)
    df["y_up"] = (df["close_next"] > df["close"]).astype("int")

    feats = ["ret1", "ret2", "ret5", "hl_spread", "co", "vol_chg1", "ma_ratio", "vol_ratio"]
    out = df[["secid", "begin", "close", "y_up"] + feats].copy()
    out = out.replace([np.inf, -np.inf], np.nan).dropna()
    return out

# период обучения
train_till = dt.date.today()
train_from = train_till - dt.timedelta(days=365 * 3)  # ~3 года

all_candles = []
for t in TICKERS:
    c = fetch_candles(t, train_from, train_till, interval=24)
    if not c.empty:
        all_candles.append(c)

candles_all = pd.concat(all_candles, ignore_index=True) if all_candles else pd.DataFrame()
if candles_all.empty:
    raise RuntimeError("Не удалось собрать свечи. Проверьте тикеры/доступность ISS.")

ds = make_features(candles_all)
print("Размер датасета:", ds.shape)
print("Доля y_up=1:", ds["y_up"].mean().round(3))

ds.sort_values(["secid", "begin"]).tail(5)

Размер датасета: (8236, 12)
Доля y_up=1: 0.486


,secid,begin,close,y_up,ret1,ret2,ret5,hl_spread,co,vol_chg1,ma_ratio,vol_ratio
8431,VTBR,2026-04-02,89.910,1,0.020140,0.044918,0.084298,0.031643,0.020603,0.442219,0.004972,0.996879
8432,VTBR,2026-04-03,91.660,1,0.019464,0.039995,0.100096,0.025966,0.019464,0.173287,0.020873,0.837426
8433,VTBR,2026-04-04,92.505,0,0.009219,0.028862,0.109306,0.013837,0.009329,-0.902939,0.038258,0.599329
8434,VTBR,2026-04-05,91.300,0,-0.013026,-0.003928,0.061073,0.021577,-0.013773,0.805532,0.047431,1.059813
8435,VTBR,2026-04-06,90.375,0,-0.010131,-0.023026,0.025416,0.019640,-0.009155,0.932021,0.050239,1.095668


In [4]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Важно: для временных рядов нельзя перемешивать данные. Сделаем простой time-based split:
# обучаемся на первых 80% дат по всем бумагам, тестируемся на последних 20%.

ds_sorted = ds.sort_values("begin").reset_index(drop=True)
cut = int(len(ds_sorted) * 0.8)
train = ds_sorted.iloc[:cut]
test = ds_sorted.iloc[cut:]

feature_cols = [c for c in ds.columns if c not in ("secid", "begin", "close", "y_up")]
X_train, y_train = train[feature_cols], train["y_up"]
X_test, y_test = test[feature_cols], test["y_up"]

clf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)
print(f"Test accuracy: {acc:.3f}")
print(f"Test ROC-AUC:  {auc:.3f}")
print(classification_report(y_test, pred, digits=3))

# Пример: оценка вероятности роста на следующий день для последней доступной точки каждой акции
latest = ds.sort_values(["secid", "begin"]).groupby("secid").tail(1)
latest = latest.copy()
latest["p_up_next_day"] = clf.predict_proba(latest[feature_cols])[:, 1]
latest[["secid", "begin", "close", "p_up_next_day"]].sort_values("p_up_next_day", ascending=False)

Test accuracy: 0.514
Test ROC-AUC:  0.508
              precision    recall  f1-score   support

           0      0.525     0.635     0.575       853
           1      0.495     0.384     0.432       795

    accuracy                          0.514      1648
   macro avg      0.510     0.510     0.504      1648
weighted avg      0.511     0.514     0.506      1648



,secid,begin,close,p_up_next_day
8435,VTBR,2026-04-06,90.375,0.541639
5061,NVTK,2026-04-06,1280.000,0.516018
4220,ROSN,2026-04-06,470.200,0.513227
5904,TATN,2026-04-06,646.000,0.485209
6749,MGNT,2026-04-06,2919.500,0.473487
2534,LKOH,2026-04-06,5591.500,0.470376
1689,GAZP,2026-04-06,133.490,0.454693
844,SBER,2026-04-06,318.970,0.439877
3375,GMKN,2026-04-06,136.800,0.435391
7594,ALRS,2026-04-06,33.820,0.434176
